In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsevaluleo";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
%sql
INSERT OVERWRITE TABLE dev.gold.product_cust_deliver
WITH Ord as(
SELECT 
    order_id,
    order_id as product_id,
    product
FROM dev.silver.orders
),
Ordq as(
    SELECT 
        order_id,
        user_id,
        eval_set,
        order_number,
        order_dow,
        order_hour_of_day,
        days_since_prior_order
    FROM dev.silver.Orderq
),
custd as(
    SELECT 
        user_id,
        name as Cust_Name,
        phone as Cust_phone,
        address as Cust_address,
        country as Cust_country
    FROM dev.silver.cust_detail
),
product as (
    SELECT 
        product_id,
        product_name,
        aisle_id,
        CAST(department_id AS string) as department_id
    FROM
    dev.silver.Product
),
propl as (
    SELECT 
        aisle_id,
        aisle
    FROM
    dev.silver.product_place
),
dep as(
    SELECT
        CAST(department_id AS string) as department_id,
        department
    FROM
    dev.silver.department
)
SELECT 
    ord.order_id,
    ordq.user_id,
    custd.Cust_Name,
    custd.Cust_phone,
    custd.Cust_address,
    CASE WHEN custd.Cust_country='CO' THEN 'COLOMBIA'
         WHEN custd.Cust_country='PE' THEN 'PERU'
         WHEN custd.Cust_country='CL' THEN 'CHILE'
         WHEN custd.Cust_country='MX' THEN 'MEXICO'
    ELSE custd.Cust_country END as Cust_country,
    product.product_id,
    product.product_name,
    dep.department,
    ordq.eval_set,
    ordq.order_number,
    ordq.order_dow,
    ordq.order_hour_of_day,
    ordq.days_since_prior_order,
    propl.aisle_id,
    propl.aisle    
FROM Ord
INNER JOIN Ordq
ON
Ord.order_id=Ordq.order_id
INNER JOIN product
ON
Ord.product_id=product.product_id
INNER JOIN custd
ON
Ordq.user_id=custd.user_id
INNER JOIN dep
ON
product.department_id=dep.department_id
INNER JOIN propl
ON
product.aisle_id=propl.aisle_id;

In [0]:
%sql
SELECT * FROM dev.gold.product_cust_deliver

CREATE SHARE SALES_DELIVERY

ALTER SHARE SALES_DELIVERY ADD dev.gold.product_cust_deliver
WITH HISTORY

CREATE RECIPIENT SALES_RECIPIENT


GRANT SELECT
ON SHARE SALES_DELIVERY
TO RECIPIENT SALES_RECIPIENT